# AI-Based Children's Mood Analysis Pipeline

This notebook consolidates the complete, production-ready AI pipeline for detecting children's facial emotions. It features:
- **YOLO11 Face Detection** with 15% bounding box margin expansion.
- **EfficientNetV2-S Classifier** with a regularized classification head (LayerNorm + GELU + Dropout).
- **Class-Weighted Cross-Entropy Loss** with **Label Smoothing** to handle FER dataset class imbalance.
- **Advanced Face Augmentations** (Lighting jitter, rotations, auto-contrast, random erasing).
- **Two-Stage Fine-Tuning** (Head warmup followed by end-to-end Cosine Annealing LR fine-tuning).
- **Temporal Prediction Smoothing (EMA)** for real-time video feeds.

In [ ]:
import os
import sys
import time
import glob
import shutil
import logging
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from tqdm.notebook import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from ultralytics import YOLO
import timm

def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')

device = get_device()
print(f"Using device: {device}")

## 1. Global Configuration

Central hyperparameter and path setup.

In [ ]:
class Config:
    CLASSES = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
    NUM_CLASSES = len(CLASSES)
    
    BASE_DIR = os.getcwd()
    DATASET_DIR = os.path.join(BASE_DIR, 'dataset')
    TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
    VAL_DIR = os.path.join(DATASET_DIR, 'val') if os.path.exists(os.path.join(DATASET_DIR, 'val')) else os.path.join(DATASET_DIR, 'test')
    TEST_DIR = os.path.join(DATASET_DIR, 'test')
    
    OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
    BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, 'best_model.pth')
    PLOT_PATH = os.path.join(OUTPUT_DIR, 'training_curves.png')
    CONFUSION_MATRIX_PATH = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
    
    YOLO_FACE_MODEL = os.path.join(BASE_DIR, 'yolo11n-face.pt')
    YOLO_FACE_HF_REPO = "AdamCodd/YOLOv11n-face-detection"
    YOLO_FACE_HF_FILE = "model.pt"
    
    MODEL_NAME = 'tf_efficientnetv2_s.in21k_ft_in1k'
    IMAGE_SIZE = 224
    DROPOUT_RATE = 0.3
    
    NORM_MEAN = [0.485, 0.456, 0.406]
    NORM_STD = [0.229, 0.224, 0.225]
    
    BATCH_SIZE = 64
    EPOCHS_HEAD = 3
    EPOCHS_FINE = 15
    LR_HEAD = 1e-3
    LR_FINE = 1e-4
    MIN_LR = 1e-6
    WEIGHT_DECAY = 1e-2
    LABEL_SMOOTHING = 0.1
    
    FACE_MARGIN = 0.15
    EMA_SMOOTHING = 0.6

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print("Configuration initialized successfully.")

## 2. Dataset Loader & Augmentations

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.labels = []
        
        if not os.path.exists(root_dir):
            raise FileNotFoundError(f"Dataset folder missing: {root_dir}")
            
        for class_idx, class_name in enumerate(Config.CLASSES):
            class_folder = os.path.join(root_dir, class_name)
            if not os.path.exists(class_folder):
                alt = [d for d in os.listdir(root_dir) if d.lower() == class_name.lower()]
                if alt:
                    class_folder = os.path.join(root_dir, alt[0])
                else:
                    continue
            
            file_paths = glob.glob(os.path.join(class_folder, "*.*"))
            valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
            for path in file_paths:
                if os.path.splitext(path.lower())[1] in valid_exts:
                    self.samples.append(path)
                    self.labels.append(class_idx)
                    
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        img_path = self.samples[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (Config.IMAGE_SIZE, Config.IMAGE_SIZE))
            
        if self.transform:
            image = self.transform(image)
        return image, label

def get_transforms():
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(Config.IMAGE_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.RandomAutocontrast(p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=Config.NORM_MEAN, std=Config.NORM_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), value='random')
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(Config.IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=Config.NORM_MEAN, std=Config.NORM_STD)
    ])
    return train_transform, val_transform

def compute_class_weights(dataset):
    counts = Counter(dataset.labels)
    total = len(dataset.labels)
    weights = [total / (counts.get(i, 1) * Config.NUM_CLASSES) for i in range(Config.NUM_CLASSES)]
    w_tensor = torch.tensor(weights, dtype=torch.float32)
    return w_tensor / w_tensor.mean()

## 3. Regularized Model Architecture

In [ ]:
class EmotionClassifier(nn.Module):
    def __init__(self, model_name=Config.MODEL_NAME, num_classes=Config.NUM_CLASSES, pretrained=True, dropout=Config.DROPOUT_RATE):
        super(EmotionClassifier, self).__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features if hasattr(self.backbone, 'num_features') else 1280
        
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)
        )
        self.legacy_classifier = None

    def forward(self, x):
        features = self.backbone(x)
        if self.legacy_classifier is not None:
            return self.legacy_classifier(features)
        return self.head(features)

    def load_state_dict_compatible(self, state_dict, device=None):
        if 'model_state_dict' in state_dict:
            state_dict = state_dict['model_state_dict']
        if any('classifier' in k for k in state_dict.keys()) and not any('head.' in k for k in state_dict.keys()):
            in_feat = self.backbone.num_features if hasattr(self.backbone, 'num_features') else 1280
            self.legacy_classifier = nn.Linear(in_feat, Config.NUM_CLASSES)
            new_state = {k.replace('backbone.classifier.', 'legacy_classifier.'): v for k, v in state_dict.items()}
            self.load_state_dict(new_state, strict=False)
        else:
            self.load_state_dict(state_dict, strict=False)

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

## 4. Training Loop & Validation

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)
    return running_loss / total, (correct / total) * 100.0

def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    f1_counts = np.zeros((Config.NUM_CLASSES, 3))
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data).item()
            total += labels.size(0)
            for p, t in zip(preds.cpu().numpy(), labels.cpu().numpy()):
                if p == t: f1_counts[t, 0] += 1
                else: f1_counts[p, 1] += 1; f1_counts[t, 2] += 1
                
    f1s = []
    for c in range(Config.NUM_CLASSES):
        tp, fp, fn = f1_counts[c]
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1s.append((2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0)
    return running_loss / total, (correct / total) * 100.0, np.mean(f1s) * 100.0

## 5. Model Evaluation & Confusion Matrix

In [ ]:
def evaluate_model(weights_path=Config.BEST_MODEL_PATH):
    if not os.path.exists(weights_path):
        print(f"No weights found at {weights_path}")
        return
    _, val_transform = get_transforms()
    test_dataset = EmotionDataset(Config.TEST_DIR, transform=val_transform)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    model = EmotionClassifier(num_classes=Config.NUM_CLASSES, pretrained=False).to(device)
    state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict_compatible(state_dict, device=device)
    model.eval()
    
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            all_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            all_labels.extend(lbls.numpy())
            
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    acc = np.mean(all_preds == all_labels) * 100.0
    print(f"Test Accuracy: {acc:.2f}%")
    
    cm = np.zeros((Config.NUM_CLASSES, Config.NUM_CLASSES), dtype=int)
    for t, p in zip(all_labels, all_preds):
        cm[t, p] += 1
        
    plt.figure(figsize=(7, 6))
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    plt.imshow(cm_norm, cmap=plt.cm.Blues)
    plt.title("Normalized Confusion Matrix")
    plt.colorbar()
    plt.xticks(range(7), Config.CLASSES, rotation=45)
    plt.yticks(range(7), Config.CLASSES)
    plt.tight_layout()
    plt.show()